In [64]:
import gymnasium as gym
import numpy as np
import pandas as pd
import random

In [65]:
epsilon = 1.0
epsilon_decay = 0.9995
min_epsilon = 0.01
num_episodes = 10000 # this number added because there are nearly 9000 iterations that can go
max_steps = 100 # max steps on this can go up to only 200, i added 100 to save the time

In [66]:
# init the environment
env = gym.make("Taxi-v4")

# action types
actions = ["south(0)", "north(1)", "east(2)", "west(3)", "pickup(4)", "dropoff(5)"]

In [67]:
# init q table (500 states x 6 actions)
q_table = np.zeros((env.observation_space.n, env.action_space.n))

# display initial q table with initial 0 values
actions = ["south(0)", "north(1)", "east(2)", "west(3)", "pickup(4)", "dropoff(5)"]
q_df = pd.DataFrame(q_table, columns=actions)
q_df.index.name = "state"
q_df


,south(0),north(1),east(2),west(3),pickup(4),dropoff(5)
state,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
495,0.0,0.0,0.0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0
497,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
# helping functions
def get_observed_q_value(reward, gamma, result_state_highest_q_value_of_action, terminated):
    if terminated:
        return reward
    return reward + (gamma * result_state_highest_q_value_of_action)

def get_td_error(observed_q_value, existing_q_value):
    return observed_q_value - existing_q_value

def update_rule_of_cell(existing_q_value, alpha, td_error):
    return existing_q_value + (alpha * td_error)

def choose_action_greedy(q_table, state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    return int(np.argmax(q_table[state]))

In [69]:
# update one cell at a time
def update_one_cell(Q, state, action, reward, next_state, terminated, alpha, gamma):
    result_state_highest_q_value_of_action = np.max(Q[next_state])
    existing_q_value = Q[state, action]

    observed_q_value = get_observed_q_value(reward, gamma, result_state_highest_q_value_of_action, terminated)
    td_error = get_td_error(observed_q_value, existing_q_value)

    Q[state, action] = update_rule_of_cell(existing_q_value, alpha, td_error)

def run_one_episode(Q, epsilon, alpha, gamma, max_steps):
    state, _ = env.reset()
    total_reward = 0

    for step in range(max_steps):
        action = choose_action_greedy(Q, state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)

        update_one_cell(Q, state, action, reward, next_state, terminated, alpha, gamma)

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return total_reward

In [70]:
# iterate over and learn
def train_q_learning(num_episodes=10000, alpha=0.9, gamma=0.95,
                     epsilon=1.0, epsilon_decay=0.9995, min_epsilon=0.01, max_steps=100):
    Q = q_table
    rewards_per_episode = []

    for episode in range(num_episodes):
        total_reward = run_one_episode(Q, epsilon, alpha, gamma, max_steps)
        rewards_per_episode.append(total_reward)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

    return Q, rewards_per_episode

In [71]:
Q, rewards = train_q_learning()

print("Non-zero cells:", np.count_nonzero(Q), "/", Q.size)
print("Average reward, first 100 episodes:", np.mean(rewards[:100]))
print("Average reward, last 100 episodes :", np.mean(rewards[-100:]))

Non-zero cells: 2400 / 3000
Average reward, first 100 episodes: -392.41
Average reward, last 100 episodes : 7.47


In [72]:
Q_df = pd.DataFrame(Q, columns=actions)
Q_df.index.name = "state"
Q_df
# print(Q_df.loc[[314, 328]].round(2))

,south(0),north(1),east(2),west(3),pickup(4),dropoff(5)
state,,,,,,
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2.752004,3.949478,2.752004,3.949478,5.209976,-5.050522
2,7.933492,9.403676,7.933492,9.403676,10.951237,0.403676
3,3.949478,5.209976,3.949478,5.209976,6.536817,-3.790024
4,-3.275187,-4.237809,-4.013807,-4.146254,-17.532265,-13.124095
...,...,...,...,...,...,...
495,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
496,5.209871,6.536817,5.209972,6.536817,-3.790027,-3.790024
497,10.951237,12.580250,10.951237,9.403676,1.950806,1.951237
